# Amazon Redshift practical: MovieLens native tables

Load MovieLens CSV files from Amazon S3 directly into native Redshift tables with COPY.

~~~text
s3://gksdatalake/bronze/movielens/movies/movies.csv
s3://gksdatalake/bronze/movielens/ratings/ratings.csv
~~~

Run Bash blocks in a terminal with AWS CLI configured and SQL blocks in Redshift Query Editor v2. This workflow does not require an external query layer or catalog.

## 1. Architecture and source verification

~~~text
movies.csv  ─┐
             ├─ S3 ── Redshift COPY ── native movielens tables
ratings.csv ─┘
~~~

Confirm the AWS identity, bucket Region, and exact objects:

~~~bash
aws sts get-caller-identity --profile training
aws s3api get-bucket-location --bucket gksdatalake --profile training

aws s3 ls s3://gksdatalake/bronze/movielens/movies/ --profile training
aws s3 ls s3://gksdatalake/bronze/movielens/ratings/ --profile training
~~~

Expected headers:

~~~csv
movieId,title,genres
~~~

~~~csv
userId,movieId,rating,timestamp
~~~

For a bucket in us-east-1, GetBucketLocation can return null. Use us-east-1 as the COPY Region in that case. CSV mode handles quoted movie titles containing commas.

## 2. Create RedshiftS3Role

For a provisioned cluster, the trust relationship allows Amazon Redshift to assume the role.

~~~bash
aws iam create-role   --role-name RedshiftS3Role   --assume-role-policy-document '{
    "Version": "2012-10-17",
    "Statement": [{
      "Effect": "Allow",
      "Principal": {"Service": "redshift.amazonaws.com"},
      "Action": "sts:AssumeRole"
    }]
  }'   --profile training
~~~

If the role already exists, inspect it instead of recreating it:

~~~bash
aws iam get-role --role-name RedshiftS3Role --profile training
~~~

## 3. Trust-policy variation for Serverless

A role used by both deployment models must trust both services.

~~~json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Principal": {
      "Service": [
        "redshift.amazonaws.com",
        "redshift-serverless.amazonaws.com"
      ]
    },
    "Action": "sts:AssumeRole"
  }]
}
~~~

Save this as redshift-s3-trust.json, then run:

~~~bash
aws iam update-assume-role-policy   --role-name RedshiftS3Role   --policy-document file://redshift-s3-trust.json   --profile training
~~~

## 4. Give the role S3 read access

Simple lab option:

~~~bash
aws iam attach-role-policy   --role-name RedshiftS3Role   --policy-arn arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess   --profile training
~~~

Preferred restricted policy:

~~~json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": "s3:ListBucket",
      "Resource": "arn:aws:s3:::gksdatalake",
      "Condition": {
        "StringLike": {
          "s3:prefix": ["bronze/movielens", "bronze/movielens/*"]
        }
      }
    },
    {
      "Effect": "Allow",
      "Action": "s3:GetObject",
      "Resource": "arn:aws:s3:::gksdatalake/bronze/movielens/*"
    }
  ]
}
~~~

Save it as movielens-s3-read-policy.json. If files use a customer-managed KMS key, also grant kms:Decrypt on that key and allow the role in its key policy.

## 5. Create and attach the restricted policy

~~~bash
ACCOUNT_ID=$(aws sts get-caller-identity   --query Account --output text --profile training)

aws iam create-policy   --policy-name RedshiftMovieLensS3Read   --policy-document file://movielens-s3-read-policy.json   --profile training

aws iam attach-role-policy   --role-name RedshiftS3Role   --policy-arn "arn:aws:iam::${ACCOUNT_ID}:policy/RedshiftMovieLensS3Read"   --profile training
~~~

Use either AmazonS3ReadOnlyAccess or the restricted policy. The restricted policy provides the ListBucket and GetObject permissions required by these COPY operations.

## 6. Obtain the role ARN and understand PassRole

~~~bash
export REDSHIFT_ROLE_ARN=$(aws iam get-role   --role-name RedshiftS3Role   --query 'Role.Arn'   --output text   --profile training)

echo "$REDSHIFT_ROLE_ARN"
~~~

The administrator attaching the role needs iam:PassRole; RedshiftS3Role itself does not.

~~~json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": "iam:PassRole",
      "Resource": "arn:aws:iam::<account-id>:role/RedshiftS3Role",
      "Condition": {
        "StringEquals": {
          "iam:PassedToService": [
            "redshift.amazonaws.com",
            "redshift-serverless.amazonaws.com"
          ]
        }
      }
    },
    {
      "Effect": "Allow",
      "Action": [
        "redshift:ModifyClusterIamRoles",
        "redshift:DescribeClusters",
        "redshift-serverless:GetNamespace",
        "redshift-serverless:ListNamespaces",
        "redshift-serverless:UpdateNamespace"
      ],
      "Resource": "*"
    }
  ]
}
~~~

## 7. Attach the role to a provisioned cluster

Creating the IAM role is not enough. Associate it with the cluster.

~~~bash
aws redshift modify-cluster-iam-roles   --cluster-identifier <redshift-cluster-name>   --add-iam-roles "$REDSHIFT_ROLE_ARN"   --default-iam-role-arn "$REDSHIFT_ROLE_ARN"   --region <aws-region>   --profile training

aws redshift describe-clusters   --cluster-identifier <redshift-cluster-name>   --query 'Clusters[0].{Status:ClusterStatus,IamRoles:IamRoles,DefaultRole:DefaultIamRoleArn}'   --region <aws-region>   --profile training
~~~

Wait until the cluster status is available. Setting the default allows IAM_ROLE default; using the explicit attached ARN also works.

## 8. Associate the role with a Serverless namespace

Inspect existing roles first:

~~~bash
aws redshift-serverless list-namespaces   --query 'namespaces[].{Name:namespaceName,Roles:iamRoles,DefaultRole:defaultIamRoleArn}'   --region <aws-region>   --profile training

aws redshift-serverless get-namespace   --namespace-name <namespace-name>   --query 'namespace.{Roles:iamRoles,DefaultRole:defaultIamRoleArn}'   --region <aws-region>   --profile training
~~~

update-namespace replaces the role list. Include every existing role plus RedshiftS3Role:

~~~bash
aws redshift-serverless update-namespace   --namespace-name <namespace-name>   --iam-roles     arn:aws:iam::<account-id>:role/<existing-role-1>     "$REDSHIFT_ROLE_ARN"   --default-iam-role-arn "$REDSHIFT_ROLE_ARN"   --region <aws-region>   --profile training
~~~

When no existing role exists, pass only "$REDSHIFT_ROLE_ARN". Do not copy the existing-role placeholder literally.

## 9. Create the native MovieLens tables

Connect to the intended Redshift database.

~~~sql
CREATE SCHEMA IF NOT EXISTS movielens;

CREATE TABLE IF NOT EXISTS movielens.movies (
    movie_id INTEGER,
    title    VARCHAR(500),
    genres   VARCHAR(500)
);

CREATE TABLE IF NOT EXISTS movielens.ratings (
    user_id          INTEGER,
    movie_id         INTEGER,
    rating           DECIMAL(2,1),
    rating_timestamp BIGINT
)
DISTSTYLE AUTO
COMPOUND SORTKEY (movie_id, user_id);
~~~

rating_timestamp stores Unix epoch seconds. It is converted to a timestamp when queried.

## 10. Grant database permissions

IAM grants access to S3; Redshift grants authorize SQL and use of the IAM role.

~~~sql
GRANT USAGE ON SCHEMA movielens TO <loader_user>;
GRANT INSERT, SELECT ON TABLE movielens.movies TO <loader_user>;
GRANT INSERT, SELECT ON TABLE movielens.ratings TO <loader_user>;

-- Only a database superuser can grant ASSUMEROLE.
GRANT ASSUMEROLE
ON 'arn:aws:iam::<account-id>:role/RedshiftS3Role'
TO <loader_user>
FOR COPY;

SELECT HAS_ASSUMEROLE_PRIVILEGE(
    '<loader_user>',
    'arn:aws:iam::<account-id>:role/RedshiftS3Role',
    'copy'
) AS can_copy_with_role;
~~~

A superuser always has ASSUMEROLE. Enabling cluster-wide fine-grained enforcement requires revoking the public default; review existing workloads before making that administrative change.

## 11. Validate both inputs with COPY NOLOAD

Replace account ID and Region. NOLOAD checks parsing and types without inserting rows.

~~~sql
COPY movielens.movies (movie_id, title, genres)
FROM 's3://gksdatalake/bronze/movielens/movies/movies.csv'
IAM_ROLE 'arn:aws:iam::<account-id>:role/RedshiftS3Role'
REGION '<aws-region>'
FORMAT AS CSV
IGNOREHEADER 1
EMPTYASNULL
BLANKSASNULL
NOLOAD;

COPY movielens.ratings (user_id, movie_id, rating, rating_timestamp)
FROM 's3://gksdatalake/bronze/movielens/ratings/ratings.csv'
IAM_ROLE 'arn:aws:iam::<account-id>:role/RedshiftS3Role'
REGION '<aws-region>'
FORMAT AS CSV
IGNOREHEADER 1
EMPTYASNULL
BLANKSASNULL
NOLOAD;
~~~

If RedshiftS3Role is associated as the default, IAM_ROLE default may replace the explicit ARN.

## 12. Load with COPY

TRUNCATE makes this a repeatable full load and affects only these native tables.

~~~sql
TRUNCATE TABLE movielens.ratings;
TRUNCATE TABLE movielens.movies;

COPY movielens.movies (movie_id, title, genres)
FROM 's3://gksdatalake/bronze/movielens/movies/movies.csv'
IAM_ROLE 'arn:aws:iam::<account-id>:role/RedshiftS3Role'
REGION '<aws-region>'
CSV IGNOREHEADER 1 EMPTYASNULL BLANKSASNULL
COMPUPDATE ON STATUPDATE ON;

COPY movielens.ratings (user_id, movie_id, rating, rating_timestamp)
FROM 's3://gksdatalake/bronze/movielens/ratings/ratings.csv'
IAM_ROLE 'arn:aws:iam::<account-id>:role/RedshiftS3Role'
REGION '<aws-region>'
CSV IGNOREHEADER 1 EMPTYASNULL BLANKSASNULL
COMPUPDATE ON STATUPDATE ON;
~~~

CSV mode correctly reads quoted titles such as "American President, The (1995)".

## 13. Validate rows and relationships

~~~sql
SELECT 'movies' AS table_name, COUNT(*) AS row_count
FROM movielens.movies
UNION ALL
SELECT 'ratings', COUNT(*)
FROM movielens.ratings
ORDER BY table_name;

SELECT * FROM movielens.movies ORDER BY movie_id LIMIT 20;
SELECT * FROM movielens.ratings ORDER BY user_id, movie_id LIMIT 20;

SELECT COUNT(*) - COUNT(DISTINCT movie_id) AS duplicate_movie_ids
FROM movielens.movies;

SELECT COUNT(*) AS ratings_without_movie
FROM movielens.ratings r
LEFT JOIN movielens.movies m ON m.movie_id = r.movie_id
WHERE m.movie_id IS NULL;

SELECT MIN(rating) AS minimum_rating,
       MAX(rating) AS maximum_rating,
       SUM(CASE WHEN rating IS NULL OR rating < 0 OR rating > 5 THEN 1 ELSE 0 END)
         AS invalid_ratings
FROM movielens.ratings;
~~~

## 14. Diagnose load failures

~~~sql
SELECT query_id, start_time, TRIM(file_name) AS file_name,
       line_number, TRIM(column_name) AS column_name,
       TRIM(column_type) AS column_type,
       TRIM(error_message) AS error_message
FROM sys_load_error_detail
ORDER BY start_time DESC
LIMIT 50;

SELECT query_id, table_name, status, start_time, end_time,
       data_source, file_format, source_file_count,
       loaded_rows, loaded_bytes, error_count
FROM sys_load_history
WHERE start_time >= DATEADD(hour, -2, GETDATE())
ORDER BY start_time DESC;
~~~

AccessDenied points to role association, trust, S3/KMS policy, bucket policy, or ASSUMEROLE. A Region error means the COPY REGION differs from the bucket. Parsing errors usually mean a header, quoting, delimiter, or column-order mismatch.

## 15. MovieLens analytics

Most-rated movies:

~~~sql
SELECT m.movie_id, m.title,
       COUNT(*) AS rating_count,
       ROUND(AVG(r.rating), 2) AS average_rating
FROM movielens.ratings r
JOIN movielens.movies m ON m.movie_id = r.movie_id
GROUP BY m.movie_id, m.title
ORDER BY rating_count DESC, average_rating DESC
LIMIT 25;
~~~

Highly rated movies with enough ratings:

~~~sql
SELECT m.movie_id, m.title,
       COUNT(*) AS rating_count,
       ROUND(AVG(r.rating), 2) AS average_rating
FROM movielens.ratings r
JOIN movielens.movies m ON m.movie_id = r.movie_id
GROUP BY m.movie_id, m.title
HAVING COUNT(*) >= 50
ORDER BY average_rating DESC, rating_count DESC
LIMIT 25;
~~~

## 16. Genres and rating time

Expand pipe-delimited genres:

~~~sql
WITH numbers(n) AS (
 SELECT 1 UNION ALL SELECT 2 UNION ALL SELECT 3 UNION ALL SELECT 4 UNION ALL
 SELECT 5 UNION ALL SELECT 6 UNION ALL SELECT 7 UNION ALL SELECT 8 UNION ALL
 SELECT 9 UNION ALL SELECT 10
),
movie_genres AS (
 SELECT m.movie_id, SPLIT_PART(m.genres, '|', n.n) AS genre
 FROM movielens.movies m CROSS JOIN numbers n
 WHERE SPLIT_PART(m.genres, '|', n.n) NOT IN ('', '(no genres listed)')
)
SELECT g.genre, COUNT(DISTINCT g.movie_id) AS movies,
       COUNT(r.rating) AS ratings, ROUND(AVG(r.rating), 2) AS average_rating
FROM movie_genres g
LEFT JOIN movielens.ratings r ON r.movie_id = g.movie_id
GROUP BY g.genre
ORDER BY ratings DESC, g.genre;
~~~

Convert epoch seconds:

~~~sql
SELECT DATE_TRUNC(
         'month',
         DATEADD(second, rating_timestamp, TIMESTAMP '1970-01-01 00:00:00')
       ) AS rating_month,
       COUNT(*) AS rating_count,
       ROUND(AVG(rating), 2) AS average_rating
FROM movielens.ratings
GROUP BY 1
ORDER BY 1;
~~~

## 17. Inspect storage and query planning

~~~sql
SELECT "schema", "table", diststyle, sortkey1,
       size AS size_mb, tbl_rows, skew_rows, unsorted, stats_off
FROM svv_table_info
WHERE "schema" = 'movielens'
ORDER BY "table";

EXPLAIN
SELECT m.title, COUNT(*) AS ratings, AVG(r.rating) AS average_rating
FROM movielens.ratings r
JOIN movielens.movies m ON m.movie_id = r.movie_id
GROUP BY m.title
ORDER BY ratings DESC
LIMIT 20;

ANALYZE movielens.movies;
ANALYZE movielens.ratings;
~~~

## 18. Optional cleanup

~~~sql
-- Removes the two native tables and the schema.
-- DROP SCHEMA movielens CASCADE;
~~~

Detach the role only if no other Redshift workload uses it:

~~~bash
aws redshift modify-cluster-iam-roles   --cluster-identifier <redshift-cluster-name>   --remove-iam-roles "$REDSHIFT_ROLE_ARN"   --region <aws-region>   --profile training
~~~

If it is the default role, choose another default before removal. Detaching the role, detaching policies, deleting customer-managed policies, and deleting the IAM role are separate administrative actions.